# Steps 4–5: Results Routing & Clinical Follow-Up Pathways

This notebook covers what happens **after** a screening result is returned:

- **Step 4 — Result routing**
  - Cervical: `NORMAL` → routine surveillance; abnormal cytology → colposcopy; `HPV_POSITIVE` → colposcopy or 1-yr repeat.
  - Lung: `RADS 1/2` → 12-month repeat LDCT; `RADS 3` → 6-month repeat; `RADS 4A/4X` → biopsy pathway.

- **Step 5 — Follow-up execution**
  - Cervical: colposcopy → CIN grade draw → LEEP / surveillance.
  - Lung: biopsy pathway with LTFU nodes at each step → malignancy confirmation → treatment.
  - Loss-to-follow-up (LTFU) modelled explicitly at each clinical decision node.

All logic lives in `followup.py`. This notebook demonstrates and tests it.

In [ ]:
import sys, random
sys.path.insert(0, '../src')

import config as cfg
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from patient import Patient
from population import sample_patient
from metrics import initialize_metrics, record_exit
from followup import (
    route_cervical_result,    # decides next step after a cervical result
    run_colposcopy,           # draws CIN grade from colposcopy
    run_treatment,            # assigns treatment based on CIN grade (with LTFU check)
    run_cervical_followup,    # full cervical pipeline: routing → colposcopy → treatment
    run_lung_followup,        # full lung pipeline: RADS routing → biopsy → treatment
)
from screening import run_screening_step, get_eligible_screenings

random.seed(cfg.RANDOM_SEED)
print('Imports OK')

## Cervical Result Routing
Each result category routes the patient to a different clinical pathway.
LTFU (loss-to-follow-up) is stochastic — run 200 trials per category to see the distribution.

In [ ]:
# All valid cervical result categories:
# Cytology:  NORMAL, ASCUS, LSIL, ASC-H, HSIL
# HPV-alone: HPV_NEGATIVE, HPV_POSITIVE
results_to_test = ['NORMAL', 'ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_NEGATIVE', 'HPV_POSITIVE']

N_TRIALS = 200  # enough to show routing probabilities given LTFU is stochastic

print(f'{"Result":<15}  Routing distribution ({N_TRIALS} trials each)')
print('-' * 75)
for result in results_to_test:
    routes = []
    for _ in range(N_TRIALS):
        p = sample_patient(0, 0, 'gynecologist', 'outpatient')
        p.age, p.has_cervix, p.cervical_result = 42, True, result
        routes.append(route_cervical_result(p, 0))
    cnt     = Counter(routes)
    summary = '  '.join(f'{k}={v}' for k, v in sorted(cnt.items()))
    print(f'  {result:<13}  {summary}')

## Colposcopy CIN Grade Distribution
For each abnormal result that triggers colposcopy, draw 1,000 CIN grades and compare
to `config.COLPOSCOPY_RESULT_PROBS`. Higher-grade triggers (HSIL) should skew toward CIN3.

In [ ]:
# These are the results that send a patient to colposcopy
triggering_results = ['ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_POSITIVE']
N = 1_000

for trigger in triggering_results:
    # Create a fresh patient for each trial — avoid state bleed between draws
    cin_counts = Counter()
    for _ in range(N):
        p = sample_patient(0, 0, 'gynecologist', 'outpatient')
        p.age, p.has_cervix, p.cervical_result = 42, True, trigger
        cin_counts[run_colposcopy(p, 0)] += 1

    expected = cfg.COLPOSCOPY_RESULT_PROBS.get(f'from_{trigger}', {})
    print(f'\nColposcopy from {trigger} (n={N:,}):')
    for grade in ['NORMAL', 'CIN1', 'CIN2', 'CIN3']:
        obs = cin_counts.get(grade, 0) / N * 100
        exp = expected.get(grade, 0) * 100
        print(f'  {grade:<8} observed={obs:5.1f}%  expected={exp:5.1f}%')

## Full Cervical Follow-Up — End-to-End Patient Traces
Run 10 patients through the complete cervical pathway: screening result → routing → colposcopy → treatment/surveillance.
Each patient's full event log is printed to verify the clinical chain.

In [ ]:
random.seed(99)
DAY     = 0
metrics = initialize_metrics()

# Collect 10 patients who are cervical-eligible and actually received a result
n_run, traced, pid = 0, [], 0
while n_run < 10:
    p = sample_patient(pid, DAY, 'gynecologist', 'outpatient')
    pid += 1
    if 'cervical' not in get_eligible_screenings(p):
        continue
    result = run_screening_step(p, 'cervical', DAY)
    if result is None:
        continue  # skipped (not due, or pre-screen LTFU)

    disposition = run_cervical_followup(p, DAY, metrics)
    p.log(DAY, f'DISPOSITION: {disposition}')
    traced.append(p)
    n_run += 1

for p in traced:
    p.print_history()

## Cervical Disposition Summary
Aggregate counts from the 10-patient trace above.

In [ ]:
from metrics import print_summary

# Populate metrics from the traced patients above
metrics['n_patients']     = len(traced)
metrics['n_eligible_any'] = len(traced)
for p in traced:
    metrics['n_screened']['cervical'] += 1
    if p.cervical_result:
        metrics['cervical_results'][p.cervical_result] += 1
    if p.exit_reason:
        record_exit(metrics, p.exit_reason)

print_summary(metrics)

## Cervical CIN Grade Distribution by Trigger (Visualization)
Stacked bar chart showing how CIN grade mix shifts as the triggering result escalates from ASCUS → HSIL.
High-grade triggers (HSIL, ASC-H) should show larger CIN2/CIN3 fractions.

In [ ]:
N = 2_000
triggers   = ['ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_POSITIVE']
cin_grades = ['NORMAL', 'CIN1', 'CIN2', 'CIN3']
colors     = ['#4CAF50', '#FFC107', '#FF5722', '#B71C1C']  # green → dark red

# For each trigger, simulate N colposcopies and record grade fractions
fractions = {}
for trigger in triggers:
    counts = Counter()
    for _ in range(N):
        p = sample_patient(0, 0, 'gynecologist', 'outpatient')
        p.age, p.has_cervix, p.cervical_result = 42, True, trigger
        counts[run_colposcopy(p, 0)] += 1
    fractions[trigger] = [counts.get(g, 0) / N for g in cin_grades]

# Stacked bar — each segment is one CIN grade
fig, ax = plt.subplots(figsize=(9, 5))
bottoms = [0] * len(triggers)
for i, grade in enumerate(cin_grades):
    vals = [fractions[t][i] * 100 for t in triggers]
    ax.bar(triggers, vals, bottom=bottoms, color=colors[i], label=grade)
    bottoms = [b + v for b, v in zip(bottoms, vals)]

ax.set_ylabel('CIN grade distribution (%)')
ax.set_title('Colposcopy CIN grade mix by triggering result')
ax.legend(title='CIN grade', loc='upper right')
ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()

## Cervical LTFU Waterfall
Simulate 1,000 patients with an abnormal result and trace how many make it through each
clinical step: abnormal result → colposcopy → treatment completion.

In [ ]:
random.seed(42)
N = 1_000

# Force all patients to start with HSIL (worst case — all should go to colposcopy)
n_abnormal  = N
n_colposcopy, n_treatment_reached, n_treated, n_surveillance, n_ltfu = 0, 0, 0, 0, 0

for _ in range(N):
    p = sample_patient(0, 0, 'gynecologist', 'outpatient')
    p.age, p.has_cervix, p.cervical_result = 42, True, 'HSIL'
    m = initialize_metrics()

    disposition = run_cervical_followup(p, 0, m)

    # Count how far each patient got
    if disposition != 'exit' or p.colposcopy_result:
        n_colposcopy += 1
    if disposition in ('treated', 'surveillance'):
        n_treatment_reached += 1
    if disposition == 'treated':
        n_treated += 1
    if disposition == 'surveillance':
        n_surveillance += 1
    if disposition == 'exit' or not p.active:
        n_ltfu += 1

# Waterfall chart
stages = ['Abnormal result', 'Reached colposcopy', 'Reached treatment step', 'Excisional tx', 'Surveillance']
counts = [n_abnormal, n_colposcopy, n_treatment_reached, n_treated, n_surveillance]
pcts   = [c / N * 100 for c in counts]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stages, pcts, color=['#1565C0', '#1976D2', '#42A5F5', '#EF5350', '#66BB6A'])
ax.set_ylabel('% of patients (starting from HSIL result)')
ax.set_title('Cervical follow-up waterfall — HSIL cohort (n=1,000)')
ax.set_ylim(0, 115)
for bar, pct, cnt in zip(bars, pcts, counts):
    ax.text(bar.get_x() + bar.get_width()/2, pct + 1.5, f'{pct:.1f}%\n(n={cnt})',
            ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

---
## Lung Follow-Up Routing
Each Lung-RADS category routes to a different next step:
- **RADS 0**: incomplete scan → repeat in 1–3 months
- **RADS 1/2**: negative/benign → routine 12-month repeat
- **RADS 3**: probably benign → 6-month repeat
- **RADS 4A/4B/4X**: suspicious → biopsy pathway (with LTFU at each step)

In [ ]:
# All Lung-RADS categories defined in config
rads_categories = list(cfg.LUNG_RADS_PROBS.keys())
N_TRIALS = 500  # stochastic LTFU — run multiple trials per category

print(f'{"RADS category":<15}  Routing distribution ({N_TRIALS} trials each)')
print('-' * 75)
for rads in rads_categories:
    dispositions = []
    for _ in range(N_TRIALS):
        p = sample_patient(0, 0, 'specialist', 'outpatient')
        p.age, p.smoker, p.pack_years = 62, True, 30
        p.lung_result = rads
        dispositions.append(run_lung_followup(p, 0))
    cnt     = Counter(dispositions)
    summary = '  '.join(f'{k}={v}' for k, v in sorted(cnt.items()))
    print(f'  {rads:<13}  {summary}')

## Lung Biopsy Pathway — LTFU Funnel
For RADS 4A and 4B/4X patients, visualize how many patients make it through each
step of the biopsy chain: result communicated → referral → scheduled → completed → malignancy confirmed → treated.

In [ ]:
random.seed(42)
N = 2_000  # use RADS_4A — highest volume of the biopsy-pathway categories

step_counts = defaultdict(int)

for _ in range(N):
    p = sample_patient(0, 0, 'specialist', 'outpatient')
    p.age, p.smoker, p.pack_years = 62, True, 30
    p.lung_result = 'RADS_4A'
    m = initialize_metrics()
    run_lung_followup(p, 0, m)

    # Tally each step using the metrics dict written by run_lung_followup
    step_counts['Result communicated']  += m.get('lung_result_communicated', 0)
    step_counts['Biopsy referral made'] += m.get('lung_biopsy_referral', 0)
    step_counts['Biopsy scheduled']     += m.get('lung_biopsy_scheduled', 0)
    step_counts['Biopsy completed']     += m.get('lung_biopsy_completed', 0)
    step_counts['Malignancy confirmed'] += m.get('lung_malignancy_confirmed', 0)
    step_counts['Treatment given']      += m.get('lung_treatment_given', 0)

# Funnel chart
steps  = ['Result communicated', 'Biopsy referral made', 'Biopsy scheduled',
          'Biopsy completed', 'Malignancy confirmed', 'Treatment given']
counts = [step_counts[s] for s in steps]
pcts   = [c / N * 100 for c in counts]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(steps[::-1], pcts[::-1],
               color=['#1565C0','#1976D2','#42A5F5','#64B5F6','#EF5350','#66BB6A'][::-1])
ax.set_xlabel('% of RADS 4A patients (n=2,000)')
ax.set_title('Lung biopsy pathway — LTFU funnel (RADS 4A cohort)')
ax.set_xlim(0, 110)
for bar, pct, cnt in zip(bars, pcts[::-1], counts[::-1]):
    ax.text(pct + 1, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%  (n={cnt})', va='center', fontsize=9)
plt.tight_layout()
plt.show()